# SSC Result Analysis


In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set Seaborn styling
sns.set_theme(style="whitegrid")

# 1. Raw Data & Save to Excel
csv_raw = """
Roll,Name,Group,Gender,Class Teacher,Result,GPA,Bangla,English,Mathematics,BGS,Physics,Chemistry,Biology,ICT,Physical Education ,Carrer Education,General Science,Agriculture Studies,Islam & Moral Ed,Geography,Civics,History
105239,Sheikh Talha,Science,Male,Dr. Jatka,PASS,4,126,136,47,62,72,71,85,43,48,48,,79,69,,,
105240,Tahia Sultana,Science,Female,Dr. Jatka,PASS,4.22,133,102,55,89,62,67,79,43,48,48,,81,83,,,
105241,Kazi Sadia Sultana Mahima,Science,Female,Dr. Jatka,PASS,4.11,116,105,61,84,67,71,77,45,48,48,,73,79,,,
105243,Toslima Begum,Science,Female,Dr. Jatka,PASS,3.78,112,94,41,82,61,64,78,44,48,48,,70,79,,,
105247,Kazi Mumina Akter Jui,Science,Female,Dr. Jatka,PASS,3.39,109,96,42,67,52,63,70,45,48,48,,69,52,,,
308975,Ariful Islam,Humanities,Male,Dr. Jatka,FAIL,0,75,0,34,,,,,37,,,39,0,0,45,0,0
308026,Md Riyad,Humanities,Male,Dr. Jatka,FAIL,0,76,0,34,,,,,37,47,47,45,55,39,34,47,45
308927,Mazedul Islam Naim,Humanities,Male,Dr. Jatka,FAIL,0,90,0,0,,,,,37,48,48,46,0,36,0,37,38
"""

df_init = pd.read_csv(io.StringIO(csv_raw))
excel_filename = "ssc_results_2026.xlsx"
df_init.to_excel(excel_filename, index=False)

# 2. Load Directly From Excel
df = pd.read_excel(excel_filename)
df.columns = df.columns.str.strip()

# Dynamically load subject list directly from Excel columns
metadata_cols = [
    "Roll",
    "Name",
    "Group",
    "Gender",
    "Class Teacher",
    "Result",
    "GPA",
]
subjects = [col for col in df.columns if col not in metadata_cols]


# 3. Grade Conversion Functions
def mark_to_grade(mark, subject):
    if pd.isna(mark):
        return np.nan
    max_mark = 200 if subject in ["Bangla", "English"] else 100
    pct = (mark / max_mark) * 100
    if pct >= 80:
        return "A+"
    elif pct >= 70:
        return "A"
    elif pct >= 60:
        return "A-"
    elif pct >= 50:
        return "B"
    elif pct >= 40:
        return "C"
    elif pct >= 33:
        return "D"
    else:
        return "F"


for subj in subjects:
    df[f"{subj}_Grade"] = df[subj].apply(lambda x: mark_to_grade(x, subj))

# Normalize scores to 100% scale
norm_scores = pd.DataFrame()
for subj in subjects:
    max_mark = 200 if subj in ["Bangla", "English"] else 100
    norm_scores[subj] = (df[subj] / max_mark) * 100

# 4. Text Analysis Output
print("=== Group Performance Summary ===")
group_summary = (
    df.groupby("Group")
    .agg(
        Total_Students=("Roll", "count"),
        Passed=("Result", lambda x: (x == "PASS").sum()),
        Failed=("Result", lambda x: (x == "FAIL").sum()),
        Mean_GPA=("GPA", "mean"),
    )
    .reset_index()
)
print(group_summary.to_string(index=False))
df
# 5. Visualizations

# Chart 1: Grade Matrix Heatmap
grades = ["A+", "A", "A-", "B", "C", "D", "F"]
grade_matrix = pd.DataFrame(
    [[df[f"{s}_Grade"].value_counts().get(g, 0) for g in grades] for s in subjects],
    index=subjects,
    columns=grades,
)

plt.figure(figsize=(10, 7))
sns.heatmap(grade_matrix, annot=True, fmt="d", cmap="Blues", cbar=True, linewidths=0.5)
plt.title("Subject-Wise Letter Grade Distribution", fontsize=14, pad=12)
plt.xlabel("Letter Grade")
plt.ylabel("Subject")
plt.tight_layout()
plt.show()

# Chart 2: Performance Ranking Bar Plot
mean_scores = norm_scores.mean().dropna().sort_values(ascending=False)
colors = [
    "#2ca02c" if s >= 60 else ("#ff7f0e" if s >= 45 else "#d62728")
    for s in mean_scores.values
]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    x=mean_scores.index,
    y=mean_scores.values,
    hue=mean_scores.index,
    palette=dict(zip(mean_scores.index, colors)),
    dodge=False,
    ax=ax,
)

if ax.get_legend():
    ax.get_legend().remove()

ref_line = ax.axhline(33, color="red", linestyle="--", label="Passing Threshold (33%)")
ax.legend(handles=[ref_line], loc="upper right")
plt.xticks(rotation=45, ha="right")
ax.set_ylabel("Mean Score (%)")
ax.set_title("Subject Performance Ranking (Average Score %)", fontsize=14, pad=12)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.1f}%",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=8,
            xytext=(0, 3),
            textcoords="offset points",
        )

plt.tight_layout()
plt.show()
df

In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set Seaborn theme
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------
# 1. RAW DATA DEFINITION & SAVE/LOAD EXCEL WORKFLOW
# ---------------------------------------------------------
csv_raw = """Column 1,Roll,Name,Group,Gender,Class Teacher,Result,GPA,Bangla,English,Mathematics,BGS,Physics,Chemistry,Biology,ICT,Physical Education ,Carrer Education,General Science,Agriculture Studies,Islam & Moral Ed,Geography,Civics,History
0,105239,Sheikh Talha,Science,Male,Dr. Jatka,PASS,4,126,136,47,62,72,71,85,43,48,48,,79,69,,,
1,105240,Tahia Sultana,Science,Female,Dr. Jatka,PASS,4.22,133,102,55,89,62,67,79,43,48,48,,81,83,,,
2,105241,Kazi Sadia Sultana Mahima,Science,Female,Dr. Jatka,PASS,4.11,116,105,61,84,67,71,77,45,48,48,,73,79,,,
3,105243,Toslima Begum,Science,Female,Dr. Jatka,PASS,3.78,112,94,41,82,61,64,78,44,48,48,,70,79,,,
4,105247,Kazi Mumina Akter Jui,Science,Female,Dr. Jatka,PASS,3.39,109,96,42,67,52,63,70,45,48,48,,69,52,,,
7,308975,Ariful Islam,Humanities,Male,Dr. Jatka,FAIL,0,75,0,34,,,,,37,,,39,0,0,45,0,0
5,308026,Md Riyad,Humanities,Male,Dr. Jatka,FAIL,0,76,0,34,,,,,37,47,47,45,55,39,34,47,45
6,308927,Mazedul Islam Naim,Humanities,Male,Dr. Jatka,FAIL,0,90,0,0,,,,,37,48,48,46,0,36,0,37,38
"""

# Save raw data to Excel
excel_input = "ssc_results_2026.xlsx"
pd.read_csv(io.StringIO(csv_raw)).to_excel(excel_input, index=False)

# Load data directly from Excel file
df = pd.read_excel(excel_input)
df.columns = df.columns.str.strip()

# Dynamically identify non-subject metadata columns vs subject columns
meta_cols = [
    "Column 1",
    "Roll",
    "Name",
    "Group",
    "Gender",
    "Class Teacher",
    "Result",
    "GPA",
]
subjects = [c for c in df.columns if c not in meta_cols]


# Grade mapping function (Bangladesh SSC 200-mark scale for Bangla/English)
def mark_to_grade(mark, subject):
    if pd.isna(mark):
        return np.nan
    max_mark = 200 if subject in ["Bangla", "English"] else 100
    pct = (mark / max_mark) * 100
    if pct >= 80:
        return "A+"
    elif pct >= 70:
        return "A"
    elif pct >= 60:
        return "A-"
    elif pct >= 50:
        return "B"
    elif pct >= 40:
        return "C"
    elif pct >= 33:
        return "D"
    else:
        return "F"


# Add grade & normalized percentage columns
for s in subjects:
    df[f"{s}_Grade"] = df.apply(lambda r: mark_to_grade(r[s], s), axis=1)
    max_m = 200 if s in ["Bangla", "English"] else 100
    df[f"{s}_Pct"] = (df[s] / max_m) * 100

# ---------------------------------------------------------
# 2. ANALYSIS CALCULATIONS (10 SECTIONS)
# ---------------------------------------------------------

# Section 1: Student Master Table
df["Total_Marks"] = df[subjects].sum(axis=1)
df["Subjects_Taken"] = df[subjects].notna().sum(axis=1)
df["Overall_Pct"] = (
    df[subjects]
    .apply(
        lambda row: [
            row[s] / (200 if s in ["Bangla", "English"] else 100) * 100
            for s in subjects
            if pd.notna(row[s])
        ],
        axis=1,
    )
    .apply(np.mean)
)

master_cols = [
    "Roll",
    "Name",
    "Group",
    "Gender",
    "Result",
    "GPA",
    "Total_Marks",
    "Overall_Pct",
    "Subjects_Taken",
]
student_master_df = df[master_cols].copy()

# Section 2: Per-Student Performance
per_student_list = []
for idx, row in df.iterrows():
    for s in subjects:
        if pd.notna(row[s]):
            per_student_list.append(
                {
                    "Roll": row["Roll"],
                    "Name": row["Name"],
                    "Group": row["Group"],
                    "Subject": s,
                    "Mark": row[s],
                    "Score (%)": round(row[f"{s}_Pct"], 1),
                    "Grade": row[f"{s}_Grade"],
                }
            )
per_student_df = pd.DataFrame(per_student_list)

# Section 3: Overall Student Statistics
overall_stats_df = pd.DataFrame(
    [
        {"Metric": "Total Students Enrolled", "Value": len(df)},
        {"Metric": "Total Passed", "Value": (df["Result"] == "PASS").sum()},
        {"Metric": "Total Failed", "Value": (df["Result"] == "FAIL").sum()},
        {
            "Metric": "Pass Rate (%)",
            "Value": round((df["Result"] == "PASS").mean() * 100, 2),
        },
        {"Metric": "Average GPA (All)", "Value": round(df["GPA"].mean(), 2)},
        {
            "Metric": "Average GPA (Passed)",
            "Value": round(df[df["Result"] == "PASS"]["GPA"].mean(), 2),
        },
        {
            "Metric": "Top Scorer (Roll / GPA)",
            "Value": f"{df.loc[df['GPA'].idxmax(), 'Name']} ({df['GPA'].max()})",
        },
    ]
)

# Section 4: Pass / Fail Analysis
pass_fail_summary = (
    df.groupby(["Group", "Gender", "Result"]).size().unstack(fill_value=0).reset_index()
)

# Section 5: Subject-wise Analysis (Weakest First)
subj_analysis = []
for s in subjects:
    marks = df[s].dropna()
    pcts = df[f"{s}_Pct"].dropna()
    grades = df[f"{s}_Grade"].dropna()
    if len(pcts) > 0:
        pass_c = (grades != "F").sum()
        fail_c = (grades == "F").sum()
        subj_analysis.append(
            {
                "Subject": s,
                "Enrolled": len(pcts),
                "Avg Mark": round(marks.mean(), 2),
                "Avg Score (%)": round(pcts.mean(), 2),
                "Highest Mark": marks.max(),
                "Lowest Mark": marks.min(),
                "Pass Count": pass_c,
                "Fail Count": fail_c,
                "Pass Rate (%)": round((pass_c / len(grades)) * 100, 2),
            }
        )
subj_analysis_df = pd.DataFrame(subj_analysis).sort_values(
    by="Avg Score (%)", ascending=True
)

# Section 6: Subject-wise Grade Distribution
grade_matrix = pd.DataFrame(
    [
        [
            df[f"{s}_Grade"].value_counts().get(g, 0)
            for g in ["A+", "A", "A-", "B", "C", "D", "F"]
        ]
        for s in subjects
    ],
    index=subjects,
    columns=["A+", "A", "A-", "B", "C", "D", "F"],
)

# Section 7: Failed Students & Failed Subjects
failed_students = []
for idx, row in df[df["Result"] == "FAIL"].iterrows():
    failed_subjs = [s for s in subjects if row[f"{s}_Grade"] == "F"]
    failed_details = [
        f"{s} ({row[s] if pd.notna(row[s]) else 0})" for s in failed_subjs
    ]
    failed_students.append(
        {
            "Roll": row["Roll"],
            "Name": row["Name"],
            "Group": row["Group"],
            "Failed Count": len(failed_subjs),
            "Failed Subjects & Marks": ", ".join(failed_details),
        }
    )
failed_df = pd.DataFrame(failed_students)

# Section 8: Student Ranking by GPA
ranking_df = df.sort_values(
    by=["GPA", "Total_Marks"], ascending=[False, False]
).reset_index(drop=True)
ranking_df["Rank"] = ranking_df.index + 1
student_ranking_df = ranking_df[
    ["Rank", "Roll", "Name", "Group", "Gender", "GPA", "Total_Marks", "Result"]
]

# Section 9: Group-wise Analysis (Gender, Science, Humanities)
group_summary_df = (
    df.groupby("Group")
    .agg(
        Total_Students=("Roll", "count"),
        Passed=("Result", lambda x: (x == "PASS").sum()),
        Failed=("Result", lambda x: (x == "FAIL").sum()),
        Pass_Rate_Pct=("Result", lambda x: round((x == "PASS").mean() * 100, 1)),
        Mean_GPA=("GPA", lambda x: round(x.mean(), 2)),
    )
    .reset_index()
)

gender_summary_df = (
    df.groupby("Gender")
    .agg(
        Total_Students=("Roll", "count"),
        Passed=("Result", lambda x: (x == "PASS").sum()),
        Failed=("Result", lambda x: (x == "FAIL").sum()),
        Pass_Rate_Pct=("Result", lambda x: round((x == "PASS").mean() * 100, 1)),
        Mean_GPA=("GPA", lambda x: round(x.mean(), 2)),
    )
    .reset_index()
)

# Section 10: Best & Weakest Subject per Student
best_weakest = []
for idx, row in df.iterrows():
    student_pcts = {s: row[f"{s}_Pct"] for s in subjects if pd.notna(row[f"{s}_Pct"])}
    if student_pcts:
        best_s = max(student_pcts, key=student_pcts.get)
        weakest_s = min(student_pcts, key=student_pcts.get)
        b_str = f"{best_s} ({student_pcts[best_s]:.1f}%)"
        w_str = f"{weakest_s} ({student_pcts[weakest_s]:.1f}%)"
    else:
        b_str, w_str = "N/A", "N/A"

    best_weakest.append(
        {
            "Roll": row["Roll"],
            "Name": row["Name"],
            "Group": row["Group"],
            "Result": row["Result"],
            "GPA": row["GPA"],
            "Best Subject": b_str,
            "Weakest Subject": w_str,
        }
    )
best_weakest_df = pd.DataFrame(best_weakest)

# ---------------------------------------------------------
# 3. CONSOLE REPORT PRINTING
# ---------------------------------------------------------
print("=" * 80)
print(" 1. STUDENT MASTER TABLE")
print("=" * 80)
print(student_master_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 3. OVERALL STUDENT STATISTICS")
print("=" * 80)
print(overall_stats_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 5. SUBJECT-WISE ANALYSIS (WEAKEST FIRST)")
print("=" * 80)
print(subj_analysis_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 7. FAILED STUDENTS & FAILED SUBJECTS")
print("=" * 80)
print(failed_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 8. STUDENT RANKING BY GPA")
print("=" * 80)
print(student_ranking_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 9. GROUP-WISE & GENDER ANALYSIS")
print("=" * 80)
print("-- Group Analysis --")
print(group_summary_df.to_string(index=False))
print("\n-- Gender Analysis --")
print(gender_summary_df.to_string(index=False))

print("\n" + "=" * 80)
print(" 10. BEST & WEAKEST SUBJECT PER STUDENT")
print("=" * 80)
print(best_weakest_df.to_string(index=False))

# ---------------------------------------------------------
# 4. EXPORT TO MULTI-TAB EXCEL WORKBOOK
# ---------------------------------------------------------
output_excel = "SSC_Comprehensive_Analysis.xlsx"
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    student_master_df.to_excel(writer, sheet_name="Student Master Table", index=False)
    per_student_df.to_excel(writer, sheet_name="Per-Student Performance", index=False)
    overall_stats_df.to_excel(writer, sheet_name="Overall Statistics", index=False)
    pass_fail_summary.to_excel(writer, sheet_name="Pass-Fail Breakdown", index=False)
    subj_analysis_df.to_excel(writer, sheet_name="Subject Analysis", index=False)
    grade_matrix.to_excel(writer, sheet_name="Grade Distribution")
    failed_df.to_excel(writer, sheet_name="Failed Details", index=False)
    student_ranking_df.to_excel(writer, sheet_name="Student Rankings", index=False)
    group_summary_df.to_excel(writer, sheet_name="Group Analysis", index=False)
    gender_summary_df.to_excel(writer, sheet_name="Gender Analysis", index=False)
    best_weakest_df.to_excel(writer, sheet_name="Best & Weakest Subject", index=False)

print(f"\n[INFO] Comprehensive multi-sheet analysis saved to '{output_excel}'.")

# ---------------------------------------------------------
# 5. GRAPHICAL VISUALIZATIONS
# ---------------------------------------------------------
# Figure 1: Grade Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(grade_matrix, annot=True, fmt="d", cmap="YlGnBu", cbar=True, linewidths=0.5)
plt.title("Subject-Wise Grade Distribution Matrix", fontsize=14, pad=12, weight="bold")
plt.xlabel("Letter Grade")
plt.ylabel("Subject")
plt.tight_layout()
plt.show()

# Figure 2: Subject Performance Ranking (Weakest to Strongest)
plt.figure(figsize=(12, 5))
colors = [
    "#d62728" if s < 40 else ("#ff7f0e" if s < 60 else "#2ca02c")
    for s in subj_analysis_df["Avg Score (%)"]
]
ax = sns.barplot(
    data=subj_analysis_df,
    x="Subject",
    y="Avg Score (%)",
    palette=colors,
    hue="Subject",
    legend=False,
)
plt.axhline(33, color="red", linestyle="--", label="Passing Threshold (33%)")
plt.xticks(rotation=45, ha="right")
plt.title(
    "Subject Performance Ranking (Average % Mark - Weakest First)",
    fontsize=14,
    pad=12,
    weight="bold",
)
plt.ylabel("Average Score (%)")
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(
            f"{h:.1f}%",
            (p.get_x() + p.get_width() / 2.0, h),
            ha="center",
            va="bottom",
            fontsize=8,
            xytext=(0, 3),
            textcoords="offset points",
        )
plt.legend()
plt.tight_layout()
plt.show()